# IAAO 102 — Flashcard & Quiz App

Interactive review-question quiz for IAAO Course 102 (Income Approach to Valuation).

All logic (grading, multiple-choice generation, attempt logging, the widget UI)
lives in `quiz_engine.py`. This notebook is just the driver: import it, run it,
view your progress.

Adding a new chapter later = drop a new `chN_flashcards.json` file in `data/`.
No code changes needed here or in `quiz_engine.py`.


## Run the quiz
Re-run the cell below any time to start a fresh attempt (pick a new chapter and/or mode).

In [1]:
import importlib, quiz_engine
importlib.reload(quiz_engine)
from quiz_engine import QuizApp

app = QuizApp()

## Progress history

Run the cell below any time to see your score trend across all logged attempts, and which questions you've missed most often overall.

In [3]:
import os
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from quiz_engine import LOG_PATH

if os.path.exists(LOG_PATH):
    log_df = pd.read_csv(LOG_PATH)
    log_df["timestamp"] = pd.to_datetime(log_df["timestamp"])
    log_df["chapter"] = log_df["chapter"].astype(str)  # "1", "2", "1-2", etc.
    log_df = log_df.sort_values("timestamp")

    # --- Score trend chart, one color per chapter/section: faint for the
    # actual scores, bold for that same group's rolling average ---
    palette = px.colors.qualitative.Plotly
    group_labels = sorted(log_df["chapter"].unique())
    color_map = {label: palette[i % len(palette)] for i, label in enumerate(group_labels)}

    fig = go.Figure()
    for group_label, group in log_df.groupby("chapter"):
        group = group.sort_values("timestamp")
        display_name = f"Section {group_label}" if "-" in group_label else f"Chapter {group_label}"
        color = color_map[group_label]

        # Actual session scores - faint
        fig.add_trace(go.Scatter(
            x=group["timestamp"], y=round(group["pct"], 1),
            mode="lines+markers", name=display_name,
            legendgroup=group_label,
            line=dict(color=color, width=1.5),
            marker=dict(color=color, size=6),
            opacity=0.35,
        ))

        # Rolling 3-session average - same color, bold - only plotted once
        # 3+ real sessions exist for that group, so it's never diluted
        if len(group) >= 3:
            rolling = group["pct"].rolling(window=3, min_periods=3).mean()
            fig.add_trace(go.Scatter(
                x=group["timestamp"], y=round(rolling, 1),
                mode="lines", name=f"(3-session avg)",
                legendgroup=group_label,
                line=dict(color=color, width=4),
                opacity=1.0,
            ))

    fig.update_layout(
        title="Quiz Score Trend Over Time",
        xaxis_title="Date", yaxis_title="Score (%)",
        yaxis_range=[0, 100],
        hovermode="x unified",
    )
    fig.show()

    # --- Most frequently missed questions (unchanged - still a list) ---
    print()
    print("Most frequently missed questions overall:")
    miss_counts = {}
    for _, row in log_df.iterrows():
        missed = str(row["missed_question_ids"]).split(";") if pd.notna(row["missed_question_ids"]) and row["missed_question_ids"] else []
        for qid in missed:
            if qid:
                miss_counts[qid] = miss_counts.get(qid, 0) + 1

    miss_df = pd.DataFrame(
        [{"question_id": qid, "times_missed": count} for qid, count in miss_counts.items()]
    ).sort_values("times_missed", ascending=False)
    display(miss_df.head(10))
else:
    print("No completed sessions logged yet - run the quiz above and finish a full deck first.")


Most frequently missed questions overall:


,question_id,times_missed
1,ch1_q2,7
3,ch1_q7,7
0,ch1_q15,4
23,ch2_q12,3
14,ch2_q2,3
13,ch2_q1,3
22,ch2_q11,3
18,ch2_q6,3
27,ch2_q22,3
29,ch2_q24,3
